# Training model

In [1]:
import pandas as pd 
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv('../datasets/new_data_sets/frame_of_colab_filt.csv')

In [3]:
df

,Unnamed: 0,movieId,title,year,content
0,0,1,Toy Story,1995,Toy Story Adventure Animation Children Comedy ...
1,1,2,Jumanji,1995,Jumanji Adventure Children Fantasy fantasy mag...
2,2,3,Grumpier Old Men,1995,Grumpier Old Men Comedy Romance moldy old A fa...
3,3,4,Waiting to Exhale,1995,Waiting to Exhale Comedy Drama Romance Cheate...
4,4,5,Father of the Bride Part II,1995,Father of the Bride Part II Comedy pregnancy r...
...,...,...,...,...,...
9514,9514,193573,Love Live! The School Idol Movie,2015,Love Live! The School Idol Movie Animation Al...
9515,9515,193581,Black Butler: Book of the Atlantic,2017,Black Butler: Book of the Atlantic Action Anim...
9516,9516,193583,No Game No Life: Zero,2017,No Game No Life: Zero Animation Comedy Fantasy...
9517,9517,193585,Flint,2017,Flint Drama A woman deals with the toxic wate...


In [17]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2),
    min_df=2,
    max_features=100000,
    lowercase=True
)
vectorizer.fit(df['content'])

,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",100000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [18]:
tfidf_matrix = vectorizer.transform(df['content'])

In [19]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 325723 stored elements and shape (9519, 34624)>

In [24]:
class Content_pipeline :

    def __init__ (self , content_df : pd.DataFrame , org_df :pd.DataFrame , tfidf_matrix ):
        self.content_df = content_df.copy()
        self.org_df = org_df
        self.tfidf_matrix = tfidf_matrix

   
    # getting titles that match with user entered title
    @staticmethod
    def matched_titles(df,inp_title : str):

        search_title = inp_title
        matches = df[
            df["title"].str.contains(
                search_title,
                case=False,
                na=False,
                regex=False
            )
        ]

        titles = matches["title"].to_numpy()
        return titles

    # getting index of title
    @staticmethod
    def get_index(df,title):

        match = df.index[    
        df['title'].str.casefold() == title.casefold()
        ].tolist()

        return match[0]

    # function to apply year filter this returns True and False values for every movie_index if movie at index is greater than year_limit it will be true other wise false
    @staticmethod
    def yearly_filtered_movs_idxs (df,limit_year : int | None):
        filtered_idxs = (
            df['year'].notna() 
            & (df['year'] >= limit_year)
        ).to_numpy()

        return filtered_idxs

    # getting similar movies
    def find_similar_movies(self , inp_title , n_recoms : int = 20 , limit_year : int | None = None):

        # creating empty recommended movies frame :
        columns = ['movieId','title','similarity_score']
        rec_movs_df = pd.DataFrame(columns=columns)

        matched_titles = self.matched_titles(self.content_df,inp_title)
        n_matched_titles = matched_titles.shape[0]

        if n_matched_titles != 0:
            
            recoms_per_matched_title = n_recoms
            print(f"number of recommendation per title : {recoms_per_matched_title}")
            for title in matched_titles :
 
                crr_idx = self.get_index(self.content_df,title)
                print(crr_idx)
                similarity = cosine_similarity(
                    self.tfidf_matrix[crr_idx],
                    self.tfidf_matrix
                ).ravel()

                # applying year filter
                if limit_year :
                    eligible = self.yearly_filtered_movs_idxs(self.content_df,limit_year=limit_year)
                    print("Similarity:", similarity.shape)
                    print("Eligible:", eligible.shape)
                    similarity[~eligible] = -1
                    
                # removing similarity of input title, from similarity and movies that are below year limit
                similarity[crr_idx] = -1
                top_indxs_per_title = similarity.argsort()[::-1][:recoms_per_matched_title]
                rec_movs = self.content_df.iloc[top_indxs_per_title][
                    ['movieId','title']    
                ]

                rec_movs['similarity_score'] = similarity[top_indxs_per_title]
                rec_movs_df = pd.concat([rec_movs_df,rec_movs],ignore_index=True)
            
        else : 
            return None
        
        rec_movs_df.drop_duplicates(subset=['movieId'],inplace=True)
        top_recommendations = rec_movs_df.sort_values(
            by='similarity_score',
            ascending=False
        ).head(n_recoms)
        
        return top_recommendations

In [25]:
my_pipeline = Content_pipeline(df,df,tfidf_matrix)

In [37]:
my_pipeline.find_similar_movies(inp_title='Lord of the',n_recoms=10,limit_year=2000)

number of recommendation per title : 10
1565
Similarity: (9519,)
Eligible: (9519,)
2566
Similarity: (9519,)
Eligible: (9519,)
2846
Similarity: (9519,)
Eligible: (9519,)
2566
Similarity: (9519,)
Eligible: (9519,)
3607
Similarity: (9519,)
Eligible: (9519,)
4102
Similarity: (9519,)
Eligible: (9519,)
4754
Similarity: (9519,)
Eligible: (9519,)


,movieId,title,similarity_score
0,4993,"Lord of the Rings: The Fellowship of the Ring,...",0.481163
1,5952,"Lord of the Rings: The Two Towers, The",0.375678
2,7153,"Lord of the Rings: The Return of the King, The",0.3217
3,118696,The Hobbit: The Battle of the Five Armies,0.210754
20,96964,"Tall Man, The",0.173812
43,71252,"Final Destination, The (Final Destination 4) (...",0.173782
62,107723,Highlander: The Search for Vengeance,0.141222
10,3409,Final Destination,0.134772
4,98809,"Hobbit: An Unexpected Journey, The",0.129285
64,31804,Night Watch (Nochnoy dozor),0.116854


In [14]:
'''
recoms = 20
matched = 3
=> 12
=> 6
=> 0
'''

'\nrecoms = 20\nmatched = 3\n=> 12\n=> 6\n=> 0\n'

In [ ]:
20//3